# SVM - Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification

In [ ]:
df = sns.load_dataset('tips')
df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
num_cols = df.select_dtypes(include="number").columns

for col in num_cols:
    plt.figure(figsize=(5, 2))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()

In [ ]:
sns.pairplot(df.corr(numeric_only=True))

In [ ]:
df.columns=df.columns.str.strip()
df.columns

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
df.hist(bins=50,figsize=(20,15))

In [ ]:
sns.set_theme(style="darkgrid")

df.hist(bins=50, figsize=(20, 15))
plt.show()

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
print(df['time'].unique())
print(df['day'].unique())
print(df['sex'].unique())
print(df['smoker'].unique())

In [ ]:
sex = pd.get_dummies(df['sex'],drop_first=True, dtype=int)
smoker = pd.get_dummies(df['smoker'],drop_first=True, dtype=int)
time = pd.get_dummies(df['time'],drop_first=True, dtype=int)
day = pd.get_dummies(df['day'],drop_first=False, dtype=int)

In [ ]:
sex

In [ ]:
day

In [ ]:
time.value_counts()

In [ ]:
smoker.value_counts()

In [ ]:
df

In [ ]:
df.drop(['sex','smoker','day','time'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df = pd.concat([
    df,
    sex, smoker, time, day
], axis=1)

In [ ]:
df.head()

In [ ]:
target_col = 'tip'

# Select only feature columns for cleaning
feature_cols = df.drop(columns=target_col)

# Compute 5-number summary for all features at once
summary = feature_cols.describe().loc[['min', '25%', '50%', '75%', 'max']]
print("5-number summary for features:")
print(summary, "\n")

# Compute IQR for all features
Q1 = feature_cols.quantile(0.25)
Q3 = feature_cols.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Create mask for all feature columns at once
mask = feature_cols.apply(lambda x: x.between(lower_fence[x.name], upper_fence[x.name]))

# Combine mask across columns: keep rows where all features are within IQR range
mask_all = mask.all(axis=1)
mask_any = mask.any(axis=1)  # keeps rows with at least one inlier feature
mask_some = mask.sum(axis=1) >= 3  # keeps rows where at least 3 features are in range

# Apply mask to original dataframe (target stays intact)
df_cleaned = df[mask_any]
# Rows that were removed (outliers)
df_dropped = df[~mask_any]  # ~ inverts the boolean mask

print("Dropped rows (outliers):")
print(df_dropped)

In [ ]:
target_col = 'tip'

# Separate features from target
feature_cols = df.drop(columns=target_col)

# Compute Q1, Q3, and IQR for all features at once
Q1 = feature_cols.quantile(0.25)
Q3 = feature_cols.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Create boolean mask for all rows at once (fully vectorized)
mask = (feature_cols >= lower_fence) & (feature_cols <= upper_fence)

# Keep only rows where all features are within fences
mask_all = mask.all(axis=1)

# Apply mask to original dataframe
df_cleaned = df[mask_all]
df_dropped = df[~mask_all]

print(f"Original rows: {df.shape[0]}")
print(f"Rows after outlier removal: {df_cleaned.shape[0]}")
print(f"Rows dropped: {df_dropped.shape[0]}")


In [ ]:
df.info()

In [ ]:
df_dropped.info()

In [ ]:
df.columns

In [ ]:
# doing the training on both datasets to see the dropping difference
X = df[['tip', 'size', 'Female', 'No', 'Dinner', 'Thur', 'Fri','Sat', 'Sun']]
y = df['total_bill']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.25,
                                                    random_state=101)

## WRONG!!!!!!!!!! u cant encode without split bleh do it again 

### Without Pipelines:

In [ ]:
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def regression_metrics_cv(model, X, y, label="Dataset", cv=5):
    """
    Compute regression metrics using cross-validation.

    Parameters:
    - model: sklearn estimator (already includes preprocessing if needed)
    - X: feature matrix
    - y: target vector
    - label: label for printing
    - cv: number of CV folds
    """
    # Cross-validated predictions
    cv_pred = cross_val_predict(model, X, y, cv=cv)
    
    # Metrics
    mse = mean_squared_error(y, cv_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, cv_pred)
    r2 = r2_score(y, cv_pred)
    
    print(f"{label} (CV {cv}-fold):")
    print(f"  R²   = {r2:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")
    
    return {'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'predictions': cv_pred}


In [ ]:
df = sns.load_dataset('tips')
df.head()
print(df['time'].unique())
print(df['day'].unique())
print(df['sex'].unique())
print(df['smoker'].unique())

In [ ]:
df.columns

In [ ]:
X = df[['tip', 'sex', 'smoker', 'day', 'time', 'size']]
y = df['total_bill']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.20,
                                                    random_state=2)

In [ ]:
X_train

**Now we will do FE, encoding, etc**

In [ ]:
sex = pd.get_dummies(X_train['sex'],drop_first=True, dtype=int)
smoker = pd.get_dummies(X_train['smoker'],drop_first=True, dtype=int)
time = pd.get_dummies(X_train['time'],drop_first=True, dtype=int)
day = pd.get_dummies(X_train['day'],drop_first=False, dtype=int)

In [ ]:
X_train

In [ ]:
X_train.drop(['sex','smoker','day','time'], axis=1, inplace=True)
X_train.head()

In [ ]:
X_train = pd.concat([
    X_train,
    sex, smoker, time, day
], axis=1)

In [ ]:
X_train.head()

In [ ]:
# now same for test data:
sex = pd.get_dummies(X_test['sex'],drop_first=True, dtype=int)
smoker = pd.get_dummies(X_test['smoker'],drop_first=True, dtype=int)
time = pd.get_dummies(X_test['time'],drop_first=True, dtype=int)
day = pd.get_dummies(X_test['day'],drop_first=False, dtype=int)
X_test.drop(['sex','smoker','day','time'], axis=1, inplace=True)
X_test.head()
X_test = pd.concat([
    X_test,
    sex, smoker, time, day
], axis=1)
X_test.head()

In [ ]:
sns.heatmap(X_train.corr())

In [ ]:
sns.heatmap(X_train.corr(),annot=True,cmap='coolwarm')

In [ ]:
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def regression_metrics_cv(model, X, y, label="Dataset", cv=5):
    """
    Compute regression metrics using cross-validation.

    Parameters:
    - model: sklearn estimator (already includes preprocessing if needed)
    - X: feature matrix
    - y: target vector
    - label: label for printing
    - cv: number of CV folds
    """
    # Cross-validated predictions
    cv_pred = cross_val_predict(model, X, y, cv=cv)
    
    # Metrics
    mse = mean_squared_error(y, cv_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, cv_pred)
    r2 = r2_score(y, cv_pred)
    
    print(f"{label} (CV {cv}-fold):")
    print(f"  R²   = {r2:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")
    
    return {'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'predictions': cv_pred}


In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train)
y_test_pred = svr_model.predict(X_test)


# Print metrics
regression_metrics(svr_model, X_train, y_train, label='Train')
regression_metrics(svr_model, X_test, y_test, label='Test')



In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on train, transform train
X_train_scaled = scaler.fit_transform(X_train)

# Transform test using the same scaler
X_test_scaled = scaler.transform(X_test)

# Optional: convert back to DataFrame for convenience
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Scaled X_train:")
print(X_train_scaled.head())

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train_scaled, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train_scaled)
y_test_pred = svr_model.predict(X_test_scaled)



# Print metrics
regression_metrics(svr_model, X_train, y_train, label='Train')
regression_metrics(svr_model, X_test, y_test, label='Test')



In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# SVR model
svr = SVR()

# Parameter grid (good ranges for SVR)
param_grid = {
    'kernel': ['rbf', 'poly'],  # RBF is default, but test others
    'C': [0.01,0.1,1,10,50,100],           # Regularization
    'gamma': ['scale', 'auto',0.01,0.1,1],  # Kernel coefficient
    'epsilon': [0.01, 0.05, 0.1, 0.2]  # Tolerance for SVR
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_grid,
    n_iter=50,              # Number of random combinations
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    n_jobs=-1,
    scoring='r2',           # Optimize for R²
    random_state=101
)

# Fit
random_search.fit(X_train_scaled, y_train)



In [ ]:

# Best estimator
best_svr = random_search.best_estimator_
print("Best SVR parameters found:")
print(random_search.best_params_)

# Predict with best estimator
y_train_pred_best = best_svr.predict(X_train_scaled)
y_test_pred_best = best_svr.predict(X_test_scaled)

# Evaluate
regression_metrics(random_search, X_train, y_train_pred_best, label='Train')
regression_metrics(random_search, X_test, y_test_pred_best, label='Test')


*All again but this time lets get the tips and not the bills*

In [ ]:
df = sns.load_dataset('tips')
df.head()
print(df['time'].unique())
print(df['day'].unique())
print(df['sex'].unique())
print(df['smoker'].unique())

In [ ]:
df.columns

In [ ]:
X = df[['total_bill', 'sex', 'smoker', 'day', 'time', 'size']]
y = df['tip']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.20,
                                                    random_state=22)

In [ ]:
X_train

**Now we will do FE, encoding, etc**

In [ ]:
sex = pd.get_dummies(X_train['sex'],drop_first=True, dtype=int)
smoker = pd.get_dummies(X_train['smoker'],drop_first=True, dtype=int)
time = pd.get_dummies(X_train['time'],drop_first=True, dtype=int)
day = pd.get_dummies(X_train['day'],drop_first=False, dtype=int)

In [ ]:
X_train

In [ ]:
X_train.drop(['sex','smoker','day','time'], axis=1, inplace=True)
X_train.head()

In [ ]:
X_train = pd.concat([
    X_train,
    sex, smoker, time, day
], axis=1)

In [ ]:
X_train.head()

In [ ]:
# now same for test data:
sex = pd.get_dummies(X_test['sex'],drop_first=True, dtype=int)
smoker = pd.get_dummies(X_test['smoker'],drop_first=True, dtype=int)
time = pd.get_dummies(X_test['time'],drop_first=True, dtype=int)
day = pd.get_dummies(X_test['day'],drop_first=False, dtype=int)
X_test.drop(['sex','smoker','day','time'], axis=1, inplace=True)
X_test.head()
X_test = pd.concat([
    X_test,
    sex, smoker, time, day
], axis=1)
X_test.head()

In [ ]:
sns.heatmap(X_train.corr())

In [ ]:
sns.heatmap(X_train.corr(),annot=True,cmap='coolwarm')

In [ ]:

def regression_metrics(y_true, y_pred, label):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{label}:")
    print(f"  R²   = {r2:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")

regression_metrics(y_train, y_train_pred, "Linear Regression (Train)")
regression_metrics(y_test, y_test_pred, "Linear Regression (Test)")

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train)
y_test_pred = svr_model.predict(X_test)

# Evaluate


# Print metrics
regression_metrics(y_train, y_train_pred, label='Train')
regression_metrics(y_test, y_test_pred, label='Test')


In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on train, transform train
X_train_scaled = scaler.fit_transform(X_train)

# Transform test using the same scaler
X_test_scaled = scaler.transform(X_test)

# Optional: convert back to DataFrame for convenience
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Scaled X_train:")
print(X_train_scaled.head())

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train_scaled, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train_scaled)
y_test_pred = svr_model.predict(X_test_scaled)

# Evaluate


# Print metrics

regression_metrics(random_search, X_train, y_train_pred_best, label='Train')
regression_metrics(random_search, X_test, y_test_pred_best, label='Test')

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# SVR model
svr = SVR()

# Parameter grid (good ranges for SVR)
param_grid = {
    'kernel': ['rbf', 'poly'],  # RBF is default, but test others
    'C': [0.01,0.1,1,10,50],           # Regularization
    'gamma': ['scale', 'auto',0.01,0.1,1],  # Kernel coefficient
    'epsilon': [0.01, 0.05, 0.1, 0.2]  # Tolerance for SVR
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_grid,
    n_iter=50,              # Number of random combinations
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    n_jobs=-1,
    scoring='r2',           # Optimize for R²
    random_state=101
)

# Fit
random_search.fit(X_train_scaled, y_train)



In [ ]:

# Best estimator
best_svr = random_search.best_estimator_
print("Best SVR parameters found:")
print(random_search.best_params_)

# Predict with best estimator
y_train_pred_best = best_svr.predict(X_train_scaled)
y_test_pred_best = best_svr.predict(X_test_scaled)

# Evaluate
regression_metrics(y_train, y_train_pred_best, label='Train (RandomCV SVR)')
regression_metrics(y_test, y_test_pred_best, label='Test (RandomCV SVR)')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)
y_test_pred = lr.predict(X_test)



In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    random_state=101
)

rf.fit(X_train, y_train)

y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)

regression_metrics(y_train, y_train_pred, "Random Forest (Train)")
regression_metrics(y_test, y_test_pred, "Random Forest (Test)")


In [ ]:
import pandas as pd

importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# Plotting the Actual vs Predicted
sns.scatterplot(x=y_test, y=y_test_pred_best, alpha=0.6)

# The "Perfect Prediction" line
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2, linestyle='--')

plt.title('Actual Tips vs. Predicted Tips (Best SVR Model)')
plt.xlabel('Actual Tip ($)')
plt.ylabel('Predicted Tip ($)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Calculate residuals
residuals = y_test - y_test_pred_best

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test_pred_best, y=residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')

plt.title('Residuals vs. Predicted Tips')
plt.xlabel('Predicted Tip ($)')
plt.ylabel('Residual (Error)')
plt.show()

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Create a pipeline that scales the features (X)
scaler_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)) # Higher C helps "reach" outliers
])

# 2. Wrap it in a TransformedTargetRegressor to scale the target (y)
model = TransformedTargetRegressor(
    regressor=scaler_pipeline,
    transformer=StandardScaler()
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# SVR model
svr = SVR()

# Parameter grid (good ranges for SVR)
param_grid = {
    # Include 'linear' - tips often have a linear relationship with total_bill
    'kernel': ['rbf', 'linear'],

    # Search a wider range of C using log space
    # Lower C (0.1 - 10) helps prevent the overfitting you saw
    'C': np.logspace(-2, 2, 10),

    # Gamma: 'scale' is usually best, but we'll test smaller values
    # to keep the RBF boundary smooth
    'gamma': ['scale', 0.001, 0.01, 0.1],

    # Epsilon: This is the "cushion" where the model ignores errors.
    # Since tips vary by dollars, 0.2 was too small.
    # Let's try up to 1.0 (ignoring $1 errors to find a better general fit).
    'epsilon': [0.1, 0.2, 0.5, 1.0]
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_grid,
    n_iter=50,              # Number of random combinations
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    n_jobs=-1,
    scoring='r2',           # Optimize for R²
    random_state=101
)

# Fit
random_search.fit(X_train_scaled, y_train)

In [ ]:

# Best estimator
best_svr = random_search.best_estimator_
print("Best SVR parameters found:")
print(random_search.best_params_)

# Predict with best estimator
y_train_pred_best = best_svr.predict(X_train_scaled)
y_test_pred_best = best_svr.predict(X_test_scaled)

# Evaluate
regression_metrics(y_train, y_train_pred_best, 'Train')
regression_metrics(y_test, y_test_pred_best, 'Test')

In [ ]:
target_col = 'tip'

# Select only feature columns for cleaning
# feature_cols = df.drop(columns=target_col)
feature_cols = X_train

# Compute 5-number summary for all features at once
summary = feature_cols.describe().loc[['min', '25%', '50%', '75%', 'max']]
print("5-number summary for features:")
print(summary, "\n")

# Compute IQR for all features
Q1 = feature_cols.quantile(0.25)
Q3 = feature_cols.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Create mask for all feature columns at once
mask = feature_cols.apply(lambda x: x.between(lower_fence[x.name], upper_fence[x.name]))

# Combine mask across columns: keep rows where all features are within IQR range
mask_all = mask.all(axis=1)
mask_any = mask.any(axis=1)  # keeps rows with at least one inlier feature
mask_some = mask.sum(axis=1) >= 3  # keeps rows where at least 3 features are in range

# Apply mask to original dataframe (target stays intact)
df_cleaned = X_train[mask_all]
# Rows that were removed (outliers)
df_dropped = X_train[~mask_all]  # ~ inverts the boolean mask

print("Dropped rows (outliers):")
print(df_dropped)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train)
y_test_pred = svr_model.predict(X_test)

# Evaluate
def regression_metrics(y_true, y_pred, dataset='Dataset'):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{dataset} Metrics:")
    print(f"R²: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

# Print metrics
regression_metrics(y_train, y_train_pred, dataset='Train')
regression_metrics(y_test, y_test_pred, dataset='Test')


In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on train, transform train
X_train_scaled = scaler.fit_transform(X_train)

# Transform test using the same scaler
X_test_scaled = scaler.transform(X_test)

# Optional: convert back to DataFrame for convenience
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Scaled X_train:")
print(X_train_scaled.head())

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Initialize SVR with RBF kernel (default)
svr_model = SVR(kernel='rbf')

# Fit the model
svr_model.fit(X_train_scaled, y_train)

# Predict on train and test
y_train_pred = svr_model.predict(X_train_scaled)
y_test_pred = svr_model.predict(X_test_scaled)

# Evaluate
def regression_metrics(y_true, y_pred, dataset='Dataset'):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{dataset} Metrics:")
    print(f"R²: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

# Print metrics
regression_metrics(y_train, y_train_pred, dataset='Train')
regression_metrics(y_test, y_test_pred, dataset='Test')


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# SVR model
svr = SVR()

# Parameter grid (good ranges for SVR)
param_grid = {
    'kernel': ['rbf', 'poly'],  # RBF is default, but test others
    'C': [0.01,0.1,1,10,50],           # Regularization
    'gamma': ['scale', 'auto',0.01,0.1,1],  # Kernel coefficient
    'epsilon': [0.01, 0.05, 0.1, 0.2]  # Tolerance for SVR
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_grid,
    n_iter=50,              # Number of random combinations
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    n_jobs=-1,
    scoring='r2',           # Optimize for R²
    random_state=101
)

# Fit
random_search.fit(X_train_scaled, y_train)



In [ ]:

# Best estimator
best_svr = random_search.best_estimator_
print("Best SVR parameters found:")
print(random_search.best_params_)

# Predict with best estimator
y_train_pred_best = best_svr.predict(X_train_scaled)
y_test_pred_best = best_svr.predict(X_test_scaled)

# Evaluate
regression_metrics(y_train, y_train_pred_best, dataset='Train (RandomCV SVR)')
regression_metrics(y_test, y_test_pred_best, dataset='Test (RandomCV SVR)')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

# Plotting the Actual vs Predicted
sns.scatterplot(x=y_test, y=y_test_pred_best, alpha=0.6)

# The "Perfect Prediction" line
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2, linestyle='--')

plt.title('Actual Tips vs. Predicted Tips (Best SVR Model)')
plt.xlabel('Actual Tip ($)')
plt.ylabel('Predicted Tip ($)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Calculate residuals
residuals = y_test - y_test_pred_best

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test_pred_best, y=residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')

plt.title('Residuals vs. Predicted Tips')
plt.xlabel('Predicted Tip ($)')
plt.ylabel('Residual (Error)')
plt.show()

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Create a pipeline that scales the features (X)
scaler_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)) # Higher C helps "reach" outliers
])

# 2. Wrap it in a TransformedTargetRegressor to scale the target (y)
model = TransformedTargetRegressor(
    regressor=scaler_pipeline,
    transformer=StandardScaler()
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# SVR model
svr = SVR()

# Parameter grid (good ranges for SVR)
param_grid = {
    # Include 'linear' - tips often have a linear relationship with total_bill
    'kernel': ['rbf', 'linear'],

    # Search a wider range of C using log space
    # Lower C (0.1 - 10) helps prevent the overfitting you saw
    'C': np.logspace(-2, 2, 10),

    # Gamma: 'scale' is usually best, but we'll test smaller values
    # to keep the RBF boundary smooth
    'gamma': ['scale', 0.001, 0.01, 0.1],

    # Epsilon: This is the "cushion" where the model ignores errors.
    # Since tips vary by dollars, 0.2 was too small.
    # Let's try up to 1.0 (ignoring $1 errors to find a better general fit).
    'epsilon': [0.1, 0.2, 0.5, 1.0]
}

# Randomized search
random_search = RandomizedSearchCV(
    estimator=svr,
    param_distributions=param_grid,
    n_iter=50,              # Number of random combinations
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    n_jobs=-1,
    scoring='r2',           # Optimize for R²
    random_state=101
)

# Fit
random_search.fit(X_train_scaled, y_train)

In [ ]:

# Best estimator
best_svr = random_search.best_estimator_
print("Best SVR parameters found:")
print(random_search.best_params_)

# Predict with best estimator
y_train_pred_best = best_svr.predict(X_train_scaled)
y_test_pred_best = best_svr.predict(X_test_scaled)

# Evaluate
regression_metrics(y_train, y_train_pred_best, dataset='Train (RandomCV SVR)')
regression_metrics(y_test, y_test_pred_best, dataset='Test (RandomCV SVR)')

### With Pipelines:

#### Tips:

In [ ]:
df = sns.load_dataset('tips')
df.head()
print(df['time'].unique())
print(df['day'].unique())
print(df['sex'].unique())
print(df['smoker'].unique())
df.columns
X = df[['total_bill', 'sex', 'smoker', 'day', 'time', 'size']]
y = df['tip']


In [ ]:
df

In [ ]:
X

In [ ]:
# =========================
# Features & target
# =========================
X = df.drop("tip", axis=1)
y = df["tip"]

# =========================
# Categorical & numeric columns
# =========================
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error, r2_score

In [ ]:
# preprocessor = ColumnTransformer(
#     transformers=[
#         ('cat',OneHotEncoder(drop='first'),categorical_cols)
#     ], remainder='passthrough'
# )

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)



In [ ]:
model = Pipeline(steps=[
    ("preprocessing",preprocessor),
    ("regressor",LinearRegression())
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.20,
                                                    random_state=42)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
# =========================
# Predictions
# =========================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# =========================
# Metrics function (your exact function)
# =========================
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def regression_metrics_cv(model, X, y, label="Dataset", cv=5):
    """
    Compute regression metrics using cross-validation.

    Parameters:
    - model: sklearn estimator (already includes preprocessing if needed)
    - X: feature matrix
    - y: target vector
    - label: label for printing
    - cv: number of CV folds
    """
    # Cross-validated predictions
    cv_pred = cross_val_predict(model, X, y, cv=cv)

    # Metrics
    mse = mean_squared_error(y, cv_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, cv_pred)
    r2 = r2_score(y, cv_pred)

    print(f"{label} (CV {cv}-fold):")
    print(f"  R²   = {r2:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  RMSE = {rmse:.4f}\n")

    return {'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'predictions': cv_pred}

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

In [ ]:
model = Pipeline(
    [
    ("preprocessing",preprocessor),
    ("svr",SVR())
    ]
)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
# =========================
# Predictions
# =========================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

In [ ]:
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

In [ ]:
# =========================
# Imports
# =========================
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# Load dataset
# =========================
tips = sns.load_dataset("tips")

# =========================
# Features & target
# =========================
X = tips.drop("tip", axis=1)
y = tips["tip"]

# =========================
# Categorical & numeric columns
# =========================
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

# =========================
# Preprocessing
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

# =========================
# Model pipeline
# =========================
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

# =========================
# Train / test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# Fit model
# =========================
model.fit(X_train, y_train)

# =========================
# Predictions
# =========================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score
import numpy as np
import seaborn as sns

# Load data
tips = sns.load_dataset("tips")
X = tips.drop("tip", axis=1)
y = tips["tip"]

# Columns
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), categorical_cols),
    ("num", StandardScaler(), numeric_cols)
])

# Full pipeline
pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("svr", SVR(kernel="rbf", C=50, gamma=0.01, epsilon=0.2))
])

# 5-fold CV
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring="r2")
print("5-fold CV R²:", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))


In [ ]:
tips = sns.load_dataset("tips")

# =========================
# Features & target
# =========================
X = tips.drop("tip", axis=1)
y = tips["tip"]

# =========================
# Categorical & numeric columns
# =========================
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

# =========================
# Preprocessing
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

# =========================
# Model pipeline
# =========================
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("svr", SVR(kernel="rbf",C=20, gamma='scale', epsilon=0.35,verbose=2,max_iter=50000))
])

# =========================
# Train / test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# Fit model
# =========================
model.fit(X_train, y_train)

# =========================
# Predictions
# =========================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

# 5-fold CV
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print("5-fold CV R²:", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.model_selection import cross_val_score

# Load dataset
tips = sns.load_dataset("tips")

# Features & target
X = tips.drop("tip", axis=1)
y = tips["tip"]

# Categorical & numeric columns
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    n_jobs=-1,
    verbose=2
)

# Model pipeline
model = Pipeline([
    ("preprocessing", preprocessor),
    ("svr", SVR(kernel="rbf", C=20, gamma='scale', epsilon=0.35))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit model
model.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# R²
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# Plot actual vs predicted
plt.figure(figsize=(12,5))

# Training data
plt.subplot(1,2,1)
plt.scatter(y_train, y_train_pred, color='blue', alpha=0.6)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.xlabel("Actual Tip")
plt.ylabel("Predicted Tip")
plt.title(f"Training Set (R² = {r2_train:.2f})")

# Test data
plt.subplot(1,2,2)
plt.scatter(y_test, y_test_pred, color='green', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Tip")
plt.ylabel("Predicted Tip")
plt.title(f"Test Set (R² = {r2_test:.2f})")

plt.tight_layout()
plt.show()
# Evaluate
def regression_metrics(y_true, y_pred, dataset='Dataset'):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{dataset} Metrics:")
    print(f"R²: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

# 5-fold CV
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print("5-fold CV R²:", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))


In [ ]:
tips = sns.load_dataset("tips")

# =========================
# Features & target
# =========================
X = tips.drop("tip", axis=1)
y = tips["tip"]

# =========================
# Categorical & numeric columns
# =========================
categorical_cols = ["sex", "smoker", "day", "time"]
numeric_cols = ["total_bill", "size"]

# =========================
# Preprocessing
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),  # scale numeric
    ]
)


# =========================
# Model pipeline
# =========================
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("svr", SVR(kernel="rbf",C=0.9, gamma='scale', epsilon=0.35,verbose=2,max_iter=50000))
])

# =========================
# Train / test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# Fit model
# =========================
model.fit(X_train, y_train)

# =========================
# Predictions
# =========================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# =========================
# Evaluate
# =========================
regression_metrics(y_train, y_train_pred, "Training Set")
regression_metrics(y_test, y_test_pred, "Test Set")

# 5-fold CV
cv_scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print("5-fold CV R²:", cv_scores)
print("Mean CV R²:", np.mean(cv_scores))

In [ ]:
from sklearn.linear_model import Lasso

model_lasso = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("lasso", Lasso(alpha=0.1, max_iter=50000))  # alpha = L1 strength
])

# Fit
model_lasso.fit(X_train, y_train)

# Predict
y_train_pred = model_lasso.predict(X_train)
y_test_pred = model_lasso.predict(X_test)

# Evaluate
regression_metrics(y_train, y_train_pred, "Training Set - Lasso")
regression_metrics(y_test, y_test_pred, "Test Set - Lasso")

# 5-fold CV
cv_scores = cross_val_score(model_lasso, X, y, cv=5, scoring="r2")
print("5-fold CV R² (Lasso):", cv_scores)
print("Mean CV R² (Lasso):", np.mean(cv_scores))
